# Supervised Learning & Model Engineering - Group project
# STEP 1: Problem framing, dataset presentation, EDA, leakage audit, split

**Dataset:** Adult / Census Income (OpenML, `name="adult"`, `version=2`)
**Group:** Clément, Eliott Monneraie, Lucas Stefani (lead for this step: Clément, reviewed by all)
**Course reference:** Session 1 (EDA, problem framing, leakage, splitting strategy)

### How to read this notebook
This notebook is our working document. Every comment is written by us, for us and for the reader of the project. We use the same labels everywhere:

| Label | Meaning |
|---|---|
| **What we see** | A fact read directly from the output above (with the number). |
| **What we decide** | A choice we commit to, with the reason. |
| **What could invalidate it** | The evidence that would make us change the decision (the course standard: what we did, why, what could invalidate it). |
| **Note for STEP X** | An idea we write down now and implement later. Nothing to do now. |
| **Limitation** | Something we cannot fix, that we will state in the report. |

### Why we do not transform the data in this notebook
In STEP 1 we only **look** at the data and **decide**. We apply only one change here: removing exact duplicate rows, because it has to happen *before* the split. Every other change (dropping a column, filling missing values, encoding categories, creating features) will be implemented in STEP 2, **inside the Pipeline** (Session 5). The reason: the model will receive raw data in production, so the transformations must be part of the fitted model object, and anything that learns from data (an imputer, an encoder) must be fitted on training folds only. If we modified the data by hand here, the Pipeline and production would not see the same transformations.

### How to run
`Runtime > Run all`. Nothing to install, Colab already has every library. Runtime: about one minute. The last cell downloads a zip with our figures for the report.

---
# Part A - Problem framing

### A.1 The business question
A **private bank** wants to identify, within a population of prospects, the people with a **high annual income**, in order to offer them **wealth-management services** (a first meeting with an advisor, an invitation to an event).
Advisors have a **limited number of calls** they can make. So we need to **rank** prospects from most to least promising, and decide where to stop.

### A.2 The ML formulation
Session 1 insists that a dataset alone does not define an ML problem: we must define the prediction before choosing a model.

| Element | Our choice |
|---|---|
| **Task** | Supervised learning, **binary classification** |
| **Target** $Y$ | $Y=1$ if annual income **> 50K USD**, $Y=0$ otherwise |
| **Model output** | a probability $\hat p(x) = P(Y=1 \mid X=x)$, not a decision |
| **Unit** | one **person** (one prospect). One row = one person. |
| **Prediction time** | when the bank has the prospect's **profile** (onboarding questionnaire / KYC), **before** it knows the actual income |
| **User** | the wealth-management sales team |
| **Decision** | contact the prospect or not: $\hat y = \mathbb{1}\{\hat p(x) \ge t\}$, with the threshold $t$ chosen later (STEP 4, Session 6) |

We keep three things separate, as in Session 1 (slide 37): the **model** produces a probability, the **log-loss** trains that model, and the **threshold** turns the probability into an operational decision.

### A.3 What success means for us
- **Predictive:** rank prospects well. Because the positive class is a minority, our main metric will be **PR-AUC** (Session 6). We also want **trustworthy probabilities** (calibration, Session 7).
- **Engineering:** a reproducible Pipeline without leakage, served by an API.
- **Business:** a false positive costs an advisor's time (a useless call). A false negative is a wealthy client we never contact (lost revenue). In private banking we consider the second error more costly.
- **Fairness:** the decision "who gets contacted" is about people and may depend on sex and race. We audit it (project requirement 10).

### A.4 Limitations we already know
- **Old data.** The data come from the **1994 US census**. 50K USD in 1994 is worth roughly twice as much today, and society has changed. A real deployment would face **distribution shift**. We use the dataset as a realistic case study, not as a production dataset.
- **Legal context.** A bank operating in France / the EU cannot target clients based on sex or race (anti-discrimination law). This drives a decision we make in Part D.

---
# Part B - Dataset presentation

**Origin.** An extraction from the US Census Bureau's *Current Population Survey* of **1994**, prepared by Barry Becker and published by Ron Kohavi (1996) on the UCI repository. It is a standard benchmark for binary classification and for **fairness** studies.

**Size.** 48,842 people, 14 explanatory variables, 1 target. About 24% of people earn more than 50K: the classes are **moderately imbalanced** (much less extreme than the 0.17% fraud rate of Session 6).

### B.1 What each column means
Some columns are not self-explanatory. Here is our glossary, which we will reuse in the report.

| Column | Meaning | Semantic type |
|---|---|---|
| `age` | age in years | numerical |
| `workclass` | type of employer: `Private` (private company), `Self-emp-not-inc` (self-employed, no company), `Self-emp-inc` (self-employed with an incorporated company, typically a business owner), `Federal-gov` / `State-gov` / `Local-gov` (public sector), `Without-pay`, `Never-worked` | categorical |
| `fnlwgt` | *final weight*: a **survey weight** computed by the Census Bureau. It says how many people of the US population this row represents. It is a property of the **sampling method**, not of the person. | numerical, not a personal characteristic |
| `education` | highest diploma, as text | ordered categorical |
| `education-num` | the same diploma coded from 1 to 16 (table below) | ordinal |
| `marital-status` | `Married-civ-spouse` (married, civilian spouse living together), `Married-AF-spouse` (spouse in the Armed Forces), `Married-spouse-absent` (married but living apart), `Divorced`, `Separated`, `Widowed`, `Never-married` | categorical |
| `occupation` | job family (`Exec-managerial`, `Prof-specialty` for professionals such as engineers or doctors, `Craft-repair`, `Sales`...) | categorical |
| `relationship` | the person's role **relative to the reference person of the household**: `Husband`, `Wife`, `Own-child` (a child of the reference person), `Not-in-family` (lives alone or with non-relatives), `Unmarried` (single parent or unmarried reference person), `Other-relative` | categorical, a **proxy for sex** (Part C.8) |
| `race` | declared race: `White`, `Black`, `Asian-Pac-Islander`, `Amer-Indian-Eskimo`, `Other` | categorical, **sensitive attribute** |
| `sex` | `Male` / `Female` | categorical, **main sensitive attribute** |
| `capital-gain` | capital gains realised during the year (selling assets above their purchase price), in USD | numerical, very skewed |
| `capital-loss` | capital losses realised during the year, in USD | numerical, very skewed |
| `hours-per-week` | usual hours worked per week | numerical |
| `native-country` | country of birth | categorical, many rare values |
| `class` (target) | `<=50K` or `>50K` | binary |

### B.2 What the `education-num` codes mean
| Code | Diploma | Code | Diploma |
|---|---|---|---|
| 1 | Preschool | 9 | HS-grad (high-school diploma) |
| 2 | 1st-4th grade | 10 | Some-college (started college, no degree) |
| 3 | 5th-6th grade | 11 | Assoc-voc (2-year vocational degree) |
| 4 | 7th-8th grade | 12 | Assoc-acdm (2-year academic degree) |
| 5 | 9th grade | 13 | Bachelors |
| 6 | 10th grade | 14 | Masters |
| 7 | 11th grade | 15 | Prof-school (law school, medical school) |
| 8 | 12th grade, no diploma | 16 | Doctorate |

Codes 1 to 8 are levels **below** a high-school diploma. We check in C.5 that this mapping is exactly one-to-one in the data.

---
# Part C - Exploratory data analysis (EDA)

### Setup: what the `CONFIG` dictionary is
The first code cell creates a Python dictionary called `CONFIG`. It stores in **one place** every number we might want to change: the random seed, the size of the test set, the number of folds, and the folders where we save files.

- **Why one place.** If we wrote `42` or `0.20` directly in five different cells and changed only one, our results would silently become inconsistent. With `CONFIG`, we change a value once.
- **What the seed is.** The split and the cross-validation folds are random. Fixing the seed (`42`) means that every time any of us runs the notebook, we get **exactly the same split**. Our numbers are therefore reproducible and comparable between us, and the professor can rerun them.
- **What happens later.** In the `.py` version of the project, this dictionary becomes a small text file, `config.yaml`. The project instructions explicitly mention YAML configuration files: settings live outside the code.

The cell also defines `savefig`, a small helper that saves each figure as a PNG in `reports/figures/`, so that we can put it in the report.

In [ ]:
# ===================== Setup & configuration =====================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold

# All the values we may want to change live here (later: config.yaml).
CONFIG = {
    "seed": 42,                      # fixed seed -> same split for everyone, every time
    "test_size": 0.20,               # 80% development / 20% test
    "n_folds": 5,                    # 5-fold StratifiedKFold inside the development set
    "fig_dir": "reports/figures",    # where we save figures for the report
    "data_dir": "data/processed",    # where we save the split
}
os.makedirs(CONFIG["fig_dir"], exist_ok=True)
os.makedirs(CONFIG["data_dir"], exist_ok=True)
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
COLORS = {"<=50K": "#8da0cb", ">50K": "#fc8d62"}

def savefig(name: str) -> None:
    # Save the current figure in reports/figures so we can use it in the report.
    path = os.path.join(CONFIG["fig_dir"], f"{name}.png")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Figure saved: {path}")

In [ ]:
# ===================== Load the data =====================
adult = fetch_openml(name="adult", version=2, as_frame=True)
df = adult.frame.copy()
TARGET = adult.target_names[0]          # name of the target column ("class")
print("Target column:", TARGET)
print("Shape        :", df.shape)
df.head()

## C.1 First look: real rows and storage types
Session 1 asks us to "know the data up close" by sampling individual rows, not only the first five.

In [ ]:
display(df.sample(8, random_state=CONFIG["seed"]))

types = pd.DataFrame({
    "storage dtype": df.dtypes.astype(str),
    "n distinct values": df.nunique(),
    "example": df.iloc[0],
})
types

**What we see.** 48,842 rows and 15 columns. Text columns are stored as `category`, the others as integers.

**What we decide.** We do not trust the storage type. `education-num` is stored as an integer but it is an **ordinal code** (B.2), and `fnlwgt` is an integer but not a characteristic of the person. As in Session 1 with Credit Default: storage type and semantic type are different concepts.

## C.2 The target
We recode the target as 0/1 (1 = `>50K`, the class we care about) and measure the **prevalence** $\pi = P(Y=1)$.

In [ ]:
print("Raw target values:", df[TARGET].unique().tolist())
y_all = (df[TARGET].astype(str).str.strip() == ">50K").astype(int)

counts = y_all.value_counts().sort_index()
prevalence = y_all.mean()
print(counts.rename({0: "<=50K (0)", 1: ">50K (1)"}))
print(f"\nPrevalence P(Y=1) = {prevalence:.2%}")
print(f"Accuracy of the naive model 'always predict 0' = {1 - prevalence:.2%}")

fig, ax = plt.subplots(figsize=(5, 3.5))
counts.rename({0: "<=50K", 1: ">50K"}).plot.bar(ax=ax, color=[COLORS["<=50K"], COLORS[">50K"]])
ax.set_title(f"Target distribution (positives = {prevalence:.1%})")
ax.set_ylabel("number of people"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=0)
savefig("01_target_distribution")
plt.show()

**What we see.** 11,687 people earn more than 50K out of 48,842: a prevalence of **23.9%**. The chart shows roughly three negatives for every positive.

**What we decide.** Accuracy is not our main metric: a useless model that always predicts "<=50K" already reaches **76.1% accuracy** while finding zero wealthy prospects (Session 6). We will focus on **PR-AUC**, whose random baseline is the prevalence (23.9%), together with precision and recall. We will also **stratify** every split so that each subset keeps 23.9% positives (Part E).

## C.3 Missing values
In the original UCI files, missing values are written as `"?"`. OpenML version 2 should have converted them to `NaN`. We check both rather than assume. The question we ask, following Session 1: is a missing value **random**, or does it carry **information**? If people with a missing `occupation` have a very different income rate, then "missing" is itself a signal.

In [ ]:
qmark = {c: int((df[c].astype(str).str.strip() == "?").sum()) for c in df.columns}
missing = pd.DataFrame({"NaN": df.isna().sum(), '"?"': pd.Series(qmark)})
missing["% NaN"] = (missing["NaN"] / len(df) * 100).round(2)
missing = missing[(missing["NaN"] > 0) | (missing['"?"'] > 0)]
print("Columns with missing values:")
display(missing)

any_na = df.isna().any(axis=1)
print(f"Rows with at least one missing value: {any_na.sum()} ({any_na.mean():.2%})")

# Is missingness informative? Positive rate when missing vs when filled in
rows = []
for c in missing.index:
    is_na = df[c].isna()
    rows.append({"column": c,
                 ">50K rate if missing": y_all[is_na].mean(),
                 ">50K rate if filled in": y_all[~is_na].mean()})
display(pd.DataFrame(rows).set_index("column").style.format("{:.1%}"))

# Are workclass and occupation missing on the same rows?
print("\nMissing workclass vs missing occupation:")
display(pd.crosstab(df["workclass"].isna(), df["occupation"].isna(),
                    rownames=["workclass missing"], colnames=["occupation missing"]))
print("workclass of the rows where only occupation is missing:")
display(df.loc[df["occupation"].isna() & df["workclass"].notna(), "workclass"]
        .value_counts().loc[lambda s: s > 0])

**What we see.**
- No `"?"` left: OpenML did convert them to `NaN`. Only three columns have missing values: `workclass` (2,799 rows, 5.7%), `occupation` (2,809 rows, 5.8%) and `native-country` (857 rows, 1.8%). In total 3,620 rows (7.4%) have at least one missing value.
- `workclass` and `occupation` are missing **on the same rows**. The 10 extra rows with a missing `occupation` all have `workclass = Never-worked`: logically, someone who never worked has no occupation. So these are not random gaps, they mostly describe people **outside employment**.
- The missingness is **informative**: when `workclass` is missing, only **9.5%** earn more than 50K, versus **24.8%** when it is filled in. For `native-country` there is no real difference (25.7% vs 23.9%).

**What we decide.** We **keep** these rows (deleting 7.4% of the data would also delete a meaningful group of people). In STEP 2, the Pipeline will fill missing categorical values with an explicit category `"Missing"`, so the model can learn that "missing" means something. We do not fill with the most frequent value (`Private`), because that would hide the signal we just found.

**What could invalidate it.** If missing values in production were caused by a different process (for example a form bug rather than unemployment), the meaning of `"Missing"` would change.

## C.4 Duplicate rows
A duplicate is a row that appears more than once. We must distinguish two very different situations:

1. **The same record entered twice.** Two rows identical on **all** columns, *including* `fnlwgt`. `fnlwgt` is a survey weight with many distinct values (it looks like 77516, 215646...). Two different people having the same age, job, diploma, hours **and** the same weight down to the unit is extremely unlikely. So such a pair is almost certainly one record copied twice.
2. **Two different people with the same profile.** Two rows identical on everything **except** `fnlwgt`. This is normal: many 25-year-old high-school graduates work 40 hours a week in sales. These are real, distinct people.

Why it matters: if the same record is copied twice, one copy can end up in the training set and the other in the test set. The model would then be tested on a row it has already seen, and the test score would be too optimistic. Session 1 calls this **group leakage** (related observations on both sides of the split).

In [ ]:
dup_full = df.duplicated().sum()
dup_wo_fnlwgt = df.drop(columns=["fnlwgt"]).duplicated().sum()
print(f"Situation 1 - identical on ALL columns (fnlwgt included): {dup_full} rows ({dup_full/len(df):.2%})")
print(f"Situation 2 - identical once fnlwgt is ignored           : {dup_wo_fnlwgt} rows ({dup_wo_fnlwgt/len(df):.2%})")

# Evidence that situation-2 rows are different people:
# identical profiles (fnlwgt and target excluded) that do NOT have the same income class
profile_cols = [c for c in df.columns if c not in ["fnlwgt", TARGET]]
conflicts = (df[profile_cols].assign(y=y_all)
             .groupby(profile_cols, observed=True, dropna=False)["y"].nunique().gt(1).sum())
print(f"\nIdentical profiles whose members have DIFFERENT income classes: {conflicts}")

print("\nExample of a situation-1 pair (same record twice):")
display(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(2))

**What we see.**
- **52 rows** (0.11%) are exact copies, `fnlwgt` included: situation 1. The example pair above is identical on all 15 columns.
- **6,374 rows** (13%) are identical once `fnlwgt` is ignored: situation 2.
- **1,029 identical profiles** contain people with **different** income classes. This proves that situation-2 rows are genuinely different people: the same record cannot be both `<=50K` and `>50K`.

**What we decide.** We **remove the 52 exact copies** before splitting (Part E), to avoid group leakage. We **keep** the situation-2 rows, because they are real people and removing them would distort the population. The impact on the data size is negligible (48,842 to 48,790 rows).

**What could invalidate it.** If the Census had legitimately sampled two different people with exactly the same weight and profile. With 52 rows out of 48,842, even in that case the effect on our results would be negligible.

## C.5 Strange codes, redundancy and extreme values
Session 1: "If a value looks strange, is it an error, a valid edge case, or evidence that your understanding of the variable is wrong?" We test five hypotheses:
1. `education` and `education-num` contain **exactly the same information**;
2. `capital-gain = 99999` is a **top-code** (a ceiling used by the Census instead of the true amount), not a real amount;
3. `hours-per-week = 99` is also a ceiling;
4. some categories are **very rare**;
5. `relationship` and `sex` are always consistent (a `Husband` should be male, a `Wife` female).

In [ ]:
# 1. education <-> education-num: one-to-one?
edu_map = (df.groupby("education", observed=True)["education-num"]
           .agg(["nunique", "min", "count"]).sort_values("min")
           .rename(columns={"nunique": "n distinct codes", "min": "code", "count": "n people"}))
display(edu_map)
one_to_one = (edu_map["n distinct codes"] == 1).all() and edu_map["code"].is_unique
print("education <-> education-num is one-to-one:", one_to_one)

# 2. and 3. ceiling values
for col, cap in [("capital-gain", 99999), ("hours-per-week", 99)]:
    mask = df[col] == cap
    print(f"\n{col} = {cap}: {mask.sum()} rows, >50K rate among them = {y_all[mask].mean():.1%}")
print(f"Largest capital-gain below 99999: {df.loc[df['capital-gain'] < 99999, 'capital-gain'].max()}")

# Ranges of numerical columns (impossible values?)
display(df.select_dtypes("number").describe().T[["min", "50%", "max"]])

# 4. rare categories (< 0.5% of rows)
cat_cols = df.select_dtypes("category").columns.drop(TARGET)
for c in cat_cols:
    vc = df[c].value_counts(normalize=True)
    print(f"{c:15s}: {df[c].nunique():3d} categories, of which {(vc < 0.005).sum():2d} are rare (<0.5%)")
print("\nTop 4 native-country:")
display(df["native-country"].value_counts(normalize=True).head(4).map("{:.1%}".format))

# 5. consistency between relationship and sex
print("\nrelationship vs sex:")
display(pd.crosstab(df["relationship"], df["sex"]))

**What we see and what we decide, hypothesis by hypothesis.**

1. **`education` vs `education-num`: confirmed redundant.** Each diploma has exactly one code and each code exactly one diploma (the table above matches B.2). The two columns carry the same information twice.
   **What we decide:** we keep **`education-num`** and drop `education` (implemented in STEP 2). Reason: `education-num` already encodes the **order** of diplomas in one column, whereas one-hot encoding `education` would create 16 columns and lose the order. Keeping both would give the model the same information twice, which also makes coefficients unstable for logistic regression (correlated predictors, Session 2).

2. **`capital-gain = 99999`: a top-code.** 244 rows have exactly 99999, and the next largest value is only 41,310: there is nothing in between. All 244 people (100%) earn more than 50K. The Census replaces very large amounts by 99999 for confidentiality.
   **What we decide:** we **keep** these values. They are a valid edge case ("very large gain, exact amount unknown"), not an error. But this perfect relation with the target is suspicious enough that we examine capital variables for leakage in Part D.

3. **`hours-per-week = 99`: also a ceiling** (137 rows, 29.9% above 50K, close to people working long hours in general). Kept as it is.

4. **Rare categories.** `native-country` has 41 countries, 38 of which represent less than 0.5% of rows each; the United States is 91.3% of the data. Other columns have at most 1 or 2 rare categories (for example `Never-worked`, 10 people; `Armed-Forces`, 15 people).
   **Note for STEP 2:** group rare categories, either `United-States` vs `Other` for `native-country` or the `min_frequency` option of the one-hot encoder, to avoid dozens of columns built on a handful of people.

5. **`relationship` vs `sex`: 4 inconsistent rows.** One `Husband` is recorded as female and three `Wife` rows as male. These are probably entry errors.
   **What we decide:** we leave them, because 4 rows cannot affect the model, and the way we recode `relationship` in Part D makes the inconsistency disappear anyway.

The numerical ranges are plausible: age from 17 to 90, hours from 1 to 99, no negative amounts.

## C.6 Distributions of the numerical variables

In [ ]:
num_cols = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, c in zip(axes.ravel(), num_cols):
    sns.histplot(df[c], bins=40, ax=ax, color="#66c2a5")
    ax.set_title(f"{c}  (skewness = {df[c].skew():.1f})")
savefig("02_numeric_distributions")
plt.show()

for c in ["capital-gain", "capital-loss"]:
    print(f"{c}: {(df[c] == 0).mean():.1%} zeros, {(df[c] > 0).mean():.1%} non-zero")
both = ((df["capital-gain"] > 0) & (df["capital-loss"] > 0)).sum()
print(f"People with BOTH a gain and a loss: {both}")
print(f"Share of people working exactly 40 hours: {(df['hours-per-week'] == 40).mean():.1%}")

**What we see in the histograms.**
- **`capital-gain` and `capital-loss`** are almost always zero (91.7% and 95.3% zeros) with a very long tail (skewness 11.9 and 4.6; a symmetric variable has a skewness close to 0). The histograms are just one bar at zero.
- **Nobody has both** a gain and a loss in the same year: the two columns are two sides of the same information.
- **`hours-per-week`** has a huge spike at 40 hours (46.7% of people): the standard full-time week.
- **`age`** is skewed to the right (median 37, from 17 to 90).
- **`fnlwgt`** varies widely, but we drop it anyway (Part D).

**Note for STEP 2 (feature engineering ideas).**
- Combine the two capital columns into one `net_capital = capital-gain - capital-loss`, since nobody has both.
- Add a 0/1 indicator `has_capital_gain`, and/or a `log1p` transform, so that a linear model is not dominated by a few huge values. Trees do not need it (they only use thresholds), but logistic regression does.
- Group `hours-per-week` into bands (part-time / full-time / overtime), because the spike at 40 makes the raw number less informative than its band (see C.7).

## C.7 How the variables relate to the target
This is not modelling yet. We want to know which variables carry signal, and whether the relations are **linear** or not, because this tells us which models should do well (Session 3).

In [ ]:
# Numerical variables: distribution by class
plot_df = df.assign(income=y_all.map({0: "<=50K", 1: ">50K"}),
                    log_capital_gain=np.log1p(df["capital-gain"]))
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, c in zip(axes, ["age", "education-num", "hours-per-week", "log_capital_gain"]):
    sns.boxplot(data=plot_df, x="income", y=c, hue="income", palette=COLORS, legend=False, ax=ax)
    ax.set_title("log(1 + capital-gain)" if c == "log_capital_gain" else c); ax.set_xlabel("")
savefig("03_numeric_vs_target")
plt.show()

# Positive rate by age band: linear or bell-shaped?
age_bins = pd.cut(df["age"], bins=[16, 25, 35, 45, 55, 65, 91])
age_rate = y_all.groupby(age_bins, observed=True).agg(["mean", "size"])
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(age_rate.index.astype(str), age_rate["mean"], marker="o", color=COLORS[">50K"])
ax.axhline(prevalence, ls="--", color="grey", label=f"average = {prevalence:.1%}")
ax.set_title(">50K rate by age band"); ax.set_ylabel(">50K rate"); ax.set_xlabel("age band"); ax.legend()
savefig("04_rate_by_age")
plt.show()
display(age_rate.rename(columns={"mean": ">50K rate", "size": "n"}).style.format({">50K rate": "{:.1%}"}))

# Positive rate by hours band
hours_bins = pd.cut(df["hours-per-week"], bins=[0, 34, 40, 49, 99],
                    labels=["<35 (part-time)", "35-40 (full-time)", "41-49", "50+ (long hours)"])
display(y_all.groupby(hours_bins, observed=True).agg(["mean", "size"])
        .rename(columns={"mean": ">50K rate", "size": "n"}).style.format({">50K rate": "{:.1%}"}))

**What we see in the boxplots** (the line in each box is the median, the box holds the middle 50% of people). People above 50K are **older** (median 43 vs 34), **more educated** (median code 12 vs 9, the box sits between Some-college and Bachelors instead of around HS-grad) and **work more hours** (their box spans 40 to 50 hours, versus 35 to 40). For `capital-gain`, both boxes are flat at zero because most people have no gain; the dots above are the minority with gains. Among `>50K`, these gains start around log 8, i.e. about 3,000 USD, and go up to the 99999 top-code.

**What we see in the age curve.** The >50K rate **rises then falls**: 1.8% for 17-25, a peak of **39.2% for 45-55**, then back to 19.8% after 65 (retirement). This is a **non-linear, bell-shaped** relation.

**Why it matters for our models.** A logistic regression has a single coefficient for age, so it can only say "older means richer" or "older means poorer", never "up then down" (Session 3, slide 4). It would need hand-made features (age bands, age squared). A decision tree finds the two thresholds by itself. We expect this to be one of the reasons tree models beat the baseline.

**Hours bands.** The rate goes from 6.9% (part-time) to 20.6% (full-time), 34.6% (41-49 h) and 42.6% (50 h and more). The relation is monotone, so bands mainly help the linear model.

**Note for STEP 2.** Age bands for the logistic regression are justified by this curve. Hours bands are an optional feature.

In [ ]:
# Categorical variables: positive rate per category (with group size)
def rate_by(col: str) -> pd.DataFrame:
    t = (pd.DataFrame({col: df[col].astype(str), "y": y_all})
         .groupby(col)["y"].agg(["mean", "size"])
         .rename(columns={"mean": ">50K rate", "size": "n"})
         .sort_values(">50K rate", ascending=False))
    return t

plot_cols = ["workclass", "marital-status", "occupation", "relationship"]
fig, axes = plt.subplots(len(plot_cols), 1, figsize=(10, 3.6 * len(plot_cols)))
for ax, c in zip(axes, plot_cols):
    t = rate_by(c)
    sns.barplot(x=t[">50K rate"], y=t.index, ax=ax, color=COLORS[">50K"])
    ax.axvline(prevalence, ls="--", color="grey", label=f"average = {prevalence:.1%}")
    ax.set_title(f">50K rate by {c}"); ax.set_xlabel(">50K rate"); ax.set_ylabel(""); ax.legend()
savefig("05_categorical_vs_target")
plt.show()

# education-num: rate by diploma level, in the natural order of the codes
edu_rate = y_all.groupby(df["education-num"]).mean()
labels = edu_map.reset_index().set_index("code")["education"]
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.bar([f"{k} {labels[k]}" for k in edu_rate.index], edu_rate.values, color=COLORS[">50K"])
ax.axhline(prevalence, ls="--", color="grey")
ax.set_title(">50K rate by education level (education-num)"); ax.set_ylabel(">50K rate")
ax.tick_params(axis="x", rotation=60)
savefig("06_rate_by_education")
plt.show()

**What we see in the categorical charts** (the grey dashed line is the average, 23.9%).
- **`marital-status`** is the strongest separator: **44.6%** of `Married-civ-spouse` earn more than 50K, versus **4.5%** of `Never-married`.
- **`relationship`** says almost the same thing: `Wife` 46.9% and `Husband` 44.9%, versus `Own-child` 1.5% (mostly young people living with their parents).
- **`occupation`**: `Exec-managerial` 47.8% and `Prof-specialty` 45.1% at the top; `Priv-house-serv` 1.2% at the bottom.
- **`workclass`**: `Self-emp-inc` (business owners) 55.3%, the highest; `Private` 21.8%.
- We also look at group sizes: `Armed-Forces` (15 people) or `Married-AF-spouse` (37 people) have rates computed on very few people, so we do not draw conclusions from them.

**What we see in the education chart.** The rate stays below 8% for every level under a high-school diploma (codes 1 to 8), then **jumps** at each degree: 15.9% at HS-grad (9), **41.3% at Bachelors (13)**, 54.9% at Masters (14), about 73% for Prof-school and Doctorate (15, 16). The relation is increasing but **not linear** (flat, then steep). This confirms that the order of `education-num` matters, which supports keeping `education-num` (C.5), and that a tree or a non-linear transformation will exploit it better than a single linear coefficient.

## C.8 Sensitive attributes: sex, race, and the `relationship` proxy
We look at sensitive attributes now, during the EDA, because the fairness audit (STEP 9) starts from here. If the **base rate** of the target differs strongly between groups, an accurate model will reproduce that gap in the decision "who gets contacted".

In [ ]:
for c in ["sex", "race"]:
    print(f"\n=== >50K rate by {c} ===")
    display(rate_by(c).style.format({">50K rate": "{:.1%}"}))

print("\n=== Share of women in each relationship category ===")
display(pd.crosstab(df["relationship"], df["sex"], normalize="index").round(3))

# Is the sex gap the same for married and non-married people?
married = np.where(df["marital-status"] == "Married-civ-spouse", "married", "not married")
gap = (pd.DataFrame({"married": married, "sex": df["sex"].astype(str), "y": y_all})
       .groupby(["married", "sex"])["y"].agg(["mean", "size"])
       .rename(columns={"mean": ">50K rate", "size": "n"}))
display(gap.style.format({">50K rate": "{:.1%}"}))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, c in zip(axes[:2], ["sex", "race"]):
    t = rate_by(c)
    sns.barplot(x=t.index, y=t[">50K rate"], ax=ax, color="#8da0cb")
    ax.axhline(prevalence, ls="--", color="grey")
    ax.set_title(f">50K rate by {c}"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
sns.barplot(data=gap.reset_index(), x="married", y=">50K rate", hue="sex",
            palette={"Female": "#e78ac3", "Male": "#8da0cb"}, ax=axes[2])
axes[2].set_title(">50K rate by sex, married vs not married"); axes[2].set_xlabel("")
savefig("07_sensitive_attributes")
plt.show()

**What we see.**
- **Sex.** 30.4% of men earn more than 50K, versus **10.9% of women**. The data have a large gap by sex (the 1994 labour market: pay gap, part-time work, career interruptions).
- **The gap is mostly a composition effect.** The third chart shows that among married people the rates are **the same** (44.8% for women, 44.6% for men). The gap comes from the fact that 89% of married people in the data are men (19,899 men vs 2,480 women), and that non-married women have a lower rate than non-married men (4.8% vs 8.2%). This is the kind of nuance we want in the report: the gap is real, but it is not "same profile, different outcome".
- **`relationship` is a proxy for sex.** `Husband` is 100% male and `Wife` 100% female (apart from the 4 inconsistent rows of C.5). `Unmarried` is 77% female. So if we remove `sex` from the inputs, the model can still recover it from `relationship`.
- **Race.** `Asian-Pac-Islander` 26.9% and `White` 25.4%, versus about 12% for `Black`, `Amer-Indian-Eskimo` and `Other`. But `Other` (406 people) and `Amer-Indian-Eskimo` (470) are small groups: in a 20% test set they shrink to about 80-95 people, so any fairness metric computed on them will be very noisy (same logic as small slices in Session 8).

The decisions that follow from these findings (which inputs the model is allowed to use, and how we treat `relationship`) are made in Part D, together with the leakage audit, because they are both questions of **which information the model may legitimately use**.

## C.9 EDA summary
Session 1 (slide 14): an EDA ends with justified decisions and explicit open assumptions, not with "the data are clean".

**What we know**
- 48,842 rows, 14 variables, prevalence 23.9%: moderate imbalance, accuracy is misleading (a naive model gets 76.1%).
- Missing values only in `workclass` (5.7%), `occupation` (5.8%) and `native-country` (1.8%). For the first two they are informative (9.5% vs 24.8% >50K).
- 52 exact duplicate rows (same record twice); 6,374 rows share a profile but are distinct people.
- `education` and `education-num` are exactly redundant.
- `capital-gain` is 91.7% zeros, top-coded at 99999 (244 rows, all >50K); nobody has both a gain and a loss.
- Strongest signals: marital status / relationship, occupation, education (jump at Bachelors), age (bell-shaped), hours worked.
- Sex gap (30.4% vs 10.9%), mostly driven by marital composition; `relationship` encodes sex for married people.

**What remains open (handled in Part D and later steps)**
- Are `capital-gain` and `capital-loss` really known at prediction time? (Part D, then checked in STEP 6 and STEP 8)
- Fairness of the model's decisions despite proxies (STEP 9).
- Representativeness: 1994 US data vs today's private-bank prospects (stated as a limitation).

---
# Part D - Leakage audit and choice of model inputs (Session 1, slides 18 to 20)
For each variable we apply the course test: **"Could we genuinely know this value at the moment the prediction is made?"**
There are three families of leakage: **target leakage** (the variable contains information about $Y$ itself), **temporal leakage** (information from after the prediction time), **group leakage** (related observations on both sides of the split).

| Variable group | Available at prediction time? | Leakage risk | Our decision |
|---|---|---|---|
| `age`, `native-country`, `marital-status`, `education-num` | Yes (declared at onboarding) | Low | **Keep.** |
| `relationship` | Yes | Low (leakage), high (fairness) | **Keep, recoded** (D.2). |
| `sex`, `race` | Yes, but legally not usable for targeting | Low (leakage), high (fairness) | **Not used as model inputs. Kept aside for the fairness audit** (D.2). |
| `education` | Yes | Low, but redundant | **Drop** (redundant with `education-num`, C.5). |
| `workclass`, `occupation`, `hours-per-week` | Yes (declared job) | Low | **Keep.** They describe the current job, which a prospect declares at onboarding. |
| `capital-gain`, `capital-loss` | **Uncertain** | **Medium** (temporal) | **Keep, under an explicit assumption, and test it** (D.1). |
| `fnlwgt` | No | Not leakage, an artefact | **Drop.** A survey weight of the Census, not a property of the person. A bank never has this value for a prospect, so a model using it could not be used in production. |
| `class` (target) | No | It is the target | Never an input. |
| Duplicate rows | Not applicable | Group leakage | 52 exact copies **removed before the split** (C.4). |

There is **no timestamp** in the data: it is a single **snapshot (cross-section) of 1994**. Like the Credit Default dataset in Session 1, we cannot prove any chronology from the row order.

## D.1 The capital-gain / capital-loss question
**The problem.** These two variables are measured **over the same year** as the income we want to predict. At the moment a bank contacts a prospect, would it know the capital gains realised this year? Probably not exactly. This is a **temporal leakage** risk: we could be using information that only becomes available at the same time as the outcome. C.5 makes the risk concrete: the 244 people with `capital-gain = 99999` are **100%** `>50K`.

**Is it target leakage (is the target built from it)?** To our knowledge, no: the Census definition of "money income" used for this target does not include capital gains. So the target is not mechanically computed from these columns. A large capital gain is a sign of **wealth** (someone who owns and sells assets), and wealthy people tend to have high incomes: it is a strong but legitimate correlation, not a copy of the target.

**Why we keep them.** In our business case, a private bank does have this kind of information: a prospect's wealth questionnaire declares their investments, and the previous year's tax return reports capital gains. We therefore make the explicit assumption: **"capital-gain and capital-loss come from the prospect's most recent declaration, available before contact."** Under this assumption the variables are legitimate, and for a wealth-management offer they are among the most relevant information.

**How and when we check that our results do not depend too much on this assumption.**
- **STEP 6 (model comparison):** we train our final model **twice**, with and without the two capital variables, using the same cross-validation. The difference in PR-AUC tells us how much of our performance relies on the assumption. We report both numbers.
- **STEP 8 (explainability):** permutation importance tells us how much the held-out performance drops when these columns are shuffled.

**What could invalidate it.** If the bank could only obtain capital gains *after* deciding whom to contact, we would have to drop them, and the "without capital" score from STEP 6 would become our honest performance.

## D.2 Sex, race and the `relationship` proxy: what goes into the model
**What we decide.**
1. **`sex` and `race` are not model inputs.** A bank operating in France / the EU cannot select whom to offer a service based on sex or race: it would be direct discrimination. We keep both columns **next to** the data, not in the inputs, because we need them to **measure** fairness in STEP 9.
2. **`relationship` is kept, but recoded.** We merge `Husband` and `Wife` into a single category `Spouse` (implemented in STEP 2). Why:
   - the useful information in `relationship` is the **household role** (being married and the main earner of a household is strongly linked to income), not the sex;
   - C.7 and C.8 support this: `Wife` (46.9%) and `Husband` (44.9%) have almost the **same** rate, so merging them loses almost no signal;
   - merging removes the most direct way for the model to read the sex from `relationship`, and also makes the 4 inconsistent rows of C.5 harmless.

**Why we do not go further now.** Removing sensitive columns ("fairness through unawareness") **does not guarantee a fair model**: sex can still be partly recovered from other variables (`Unmarried` is 77% female, and `occupation` and `hours-per-week` also differ by sex). We cannot know how strong this indirect effect is **until we have a trained model**. The course and the project instructions ask for something precise: measure fairness metrics on the model's decisions, apply **one mitigation** technique, and compare the gap **before and after** on the untouched test set. That requires a model, so it belongs to **STEP 9**. In STEP 1 we only remove what is clearly illegitimate, and we write down the proxy risk.

**Which attribute for the audit.** `sex` is our **main** sensitive attribute: two large groups (32,650 men, 16,192 women), so the metrics are stable, and the mitigation will be applied on it. `race` is **secondary**: we will report metrics for each group with its size, and we will say explicitly that the small groups (`Other`, `Amer-Indian-Eskimo`) give unstable estimates. We do not merge all non-white groups into one, because `Asian-Pac-Islander` has a higher rate than `White` while `Black` has a much lower one: merging them would hide opposite effects.

**What could invalidate it.** If removing `sex` and recoding `relationship` cost a lot of performance (we will see it in the STEP 9 comparison), the trade-off would have to be discussed explicitly. A bank could not use `sex` legally anyway, so the "with sex" model would only be a reference point, not an option.

---
# Part E - Splitting strategy (Session 1, slides 21 to 24)

### E.1 Vocabulary: what is a split, a fold, K-fold, StratifiedKFold?
- **Train / test split.** We cut the data in two. The **development set** (80%) is everything we are allowed to look at and learn from. The **test set** (20%) is locked away and used **once**, at the very end (STEP 6), to report the performance of choices that are already frozen. If we used the test set to choose anything, its score would no longer be an honest estimate (Session 1, slide 8).
- **Validation and folds.** Inside the development set, we still need to compare models and choose hyperparameters without touching the test set. We set aside a part of the development set to validate. Each part is called a **fold**.
- **K-fold cross-validation.** Instead of using one single validation part, we cut the development set into **K equal folds** (we use K = 5). We train 5 times: each time on 4 folds, and we validate on the remaining fold. Each row is used for validation exactly once. The 5 scores give a mean (performance) and a standard deviation (how much the score depends on the particular sample).

```
Development set cut into 5 folds:
  run 1:  [VAL] [train] [train] [train] [train]
  run 2:  [train] [VAL] [train] [train] [train]
  run 3:  [train] [train] [VAL] [train] [train]
  run 4:  [train] [train] [train] [VAL] [train]
  run 5:  [train] [train] [train] [train] [VAL]
  -> 5 validation scores -> mean +/- standard deviation
```

- **Stratified.** "Stratified" means each fold keeps the **same proportion of positives** as the whole data (23.9%). Without it, one fold could contain 20% positives and another 28% by chance, and the scores would vary for a reason unrelated to the model. `StratifiedKFold` is the scikit-learn tool that does K-fold with this guarantee.
- **TimeSeriesSplit** (which we do **not** use). It is for data ordered in time: it always trains on the past and validates on the future. Our data have no time order between people (a 1994 snapshot), so using the row order as a chronology would invent a structure that does not exist (same reasoning as Session 1, slide 24).
- **GroupKFold** (which we do **not** use). It is for data where several rows belong to the same entity (same customer, same household): it keeps all rows of one entity in the same fold. We have no person or household identifier, and we removed the exact duplicates instead.

### E.2 What we decide
- **80% development / 20% test, stratified** on the target, with `random_state = 42` from `CONFIG`.
- **5-fold `StratifiedKFold`** inside the development set for every later choice: model comparison, hyperparameters, decision threshold, calibrator, fairness mitigation.
- The test set is **used once**, in STEP 6.
- **What stratification does not protect against** (Session 1, slide 21): temporal, group, target or preprocessing leakage. That is why we handled duplicates (C.4) and capital variables (D.1) separately, and why all preprocessing will live inside the Pipeline (STEP 2).
- **Note:** the OpenML file merges the historical UCI train and test files. We make **our own split** to control the stratification and the seed.

In [ ]:
# ===================== Remove exact duplicates, then split =====================
n_before = len(df)
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f"Exact duplicates removed: {n_before - len(df_clean)} -> {len(df_clean)} rows")

# X / y. We keep ALL raw columns here: dropping fnlwgt / education, setting sex and race aside
# and recoding relationship are done INSIDE the Pipeline in STEP 2 (see the note at the top).
y = (df_clean[TARGET].astype(str).str.strip() == ">50K").astype(int).rename("target")
X = df_clean.drop(columns=[TARGET])

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=CONFIG["test_size"], stratify=y, random_state=CONFIG["seed"])

summary = pd.DataFrame({
    "n rows": [len(y), len(y_dev), len(y_test)],
    "prevalence": [y.mean(), y_dev.mean(), y_test.mean()],
}, index=["total", "development (80%)", "test (20%)"])
display(summary.style.format({"prevalence": "{:.2%}"}))

# Save the split. Parquet keeps the 'category' types, unlike CSV.
X_dev.assign(target=y_dev).to_parquet(os.path.join(CONFIG["data_dir"], "dev.parquet"))
X_test.assign(target=y_test).to_parquet(os.path.join(CONFIG["data_dir"], "test.parquet"))
print("Saved in", CONFIG["data_dir"])

In [ ]:
# ===================== Cross-validation plan (development set only) =====================
skf = StratifiedKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=CONFIG["seed"])
rows = []
for k, (tr, va) in enumerate(skf.split(X_dev, y_dev), start=1):
    rows.append({"fold": k, "n train": len(tr), "n validation": len(va),
                 "prevalence train": y_dev.iloc[tr].mean(),
                 "prevalence validation": y_dev.iloc[va].mean()})
display(pd.DataFrame(rows).set_index("fold")
        .style.format({"prevalence train": "{:.2%}", "prevalence validation": "{:.2%}"}))

**What we see.** After removing the 52 duplicates we have 48,790 rows: 39,032 in development and 9,758 in test. The prevalence is the same (about 23.9%) in the total, the development set, the test set and each of the 5 validation folds (each fold has about 7,800 rows). The stratification works as intended.

---
# Part F - Decisions log for STEP 1 (feeds DECISIONS.md and the report)

| # | Decision | Where it is implemented |
|---|---|---|
| 1 | Target: `>50K` = 1. Main metric PR-AUC (baseline = prevalence 23.9%), not accuracy. | STEP 3 |
| 2 | Remove 52 exact duplicate rows before the split (group leakage). Keep same-profile rows (distinct people). | Here (Part E) |
| 3 | Drop `fnlwgt` (survey weight, not available in production). | STEP 2 Pipeline |
| 4 | Drop `education`, keep `education-num` (exact redundancy, order preserved). | STEP 2 Pipeline |
| 5 | Missing categorical values become an explicit `"Missing"` category (missingness is informative). | STEP 2 Pipeline |
| 6 | Keep `capital-gain = 99999` as a valid top-code. | No change |
| 7 | Keep capital variables under the assumption "known from the last declaration"; quantify the dependence with a with/without comparison. | STEP 6 and STEP 8 |
| 8 | `sex` and `race` are not model inputs; kept aside for the fairness audit. | STEP 2 Pipeline, STEP 9 |
| 9 | `relationship`: merge `Husband` and `Wife` into `Spouse`. | STEP 2 Pipeline |
| 10 | Fairness audit: `sex` main attribute (mitigation), `race` secondary (per-group with sizes). | STEP 9 |
| 11 | 80/20 stratified split, seed 42; 5-fold StratifiedKFold in development; test used once. | Here (Part E), then all steps |

**Notes for STEP 2 (feature engineering candidates):** `net_capital = capital-gain - capital-loss`; `has_capital_gain` indicator and/or `log1p`; age bands (for the linear model, justified by the bell curve); hours bands; group rare `native-country` values.

In [ ]:
# ===================== Download the figures (Colab) =====================
# Colab deletes files when the session closes: we zip the figures to keep them.
import shutil
shutil.make_archive("step1_figures", "zip", CONFIG["fig_dir"])
try:
    from google.colab import files
    files.download("step1_figures.zip")
except ImportError:
    print("Not on Colab: the zip is in the current folder -> step1_figures.zip")